# 🚀 Model Deployment – Predict Immunotherapy Response
## Hackathon DIGPHAT – PharmacogenomicDay

In this notebook you will:
1. **Load** the trained model and preprocessing artifacts saved from the Development Notebook
2. **Load** the held-out test data (unseen during training)
3. **Apply** the exact same preprocessing pipeline
4. **Generate** predictions (probabilities + binary labels)
5. **Submit** your predictions via Google Form for evaluation

In [ ]:
import pandas as pd
import numpy as np
import os, joblib, warnings, requests
warnings.filterwarnings('ignore')

print("Imports OK ✅")

---
# 1. Load Model & Preprocessing Artifacts

All fitted objects (model, scalers, PCA, selected features, imputer) were saved
in the Development Notebook. We load them here to ensure the **exact same**
transformations are applied to the test data.

In [ ]:
# ── Load saved artifacts ──────────────────────────────────────────────────────
artifacts = joblib.load('artifacts/pipeline_artifacts.joblib')

model            = artifacts['model']
model_name       = artifacts['model_name']
trans_scaler     = artifacts['trans_scaler']
pathway_scaler   = artifacts['pathway_scaler']
clinical_scaler  = artifacts['clinical_scaler']
pca              = artifacts['pca']
keep_genes       = artifacts['keep_genes']
mutation_cols    = artifacts['mutation_cols']
cell_type_cols   = artifacts['cell_type_cols']
quality_cols     = artifacts['quality_cols']
clinical_encoded_cols = artifacts['clinical_encoded_cols']
num_cols         = artifacts['num_cols']
cat_cols         = artifacts['cat_cols']
clinical_feature_cols = artifacts['clinical_feature_cols']
selected_small_cols   = artifacts['selected_small_cols']
N_COMPONENTS     = artifacts['n_pca_components']
mice_kernel      = artifacts['mice_kernel']
l1_selector      = artifacts['l1_selector']

print(f"✅ Loaded model: {model_name}")
print(f"   Expected features: {len(selected_small_cols) + N_COMPONENTS}")

---
# 2. Load External Test Data

The test data was saved per modality in `Data/test/` during the Development Notebook.
**No labels are provided** — this is a blind evaluation.

In [ ]:
# ── Load test modalities ──────────────────────────────────────────────────────
test_clin = pd.read_csv('Data/test/clinical.csv')
test_gen  = pd.read_csv('Data/test/genomic.csv')
test_dec  = pd.read_csv('Data/test/cell_deconvolution.csv')
test_ssg  = pd.read_csv('Data/test/ssgsea.csv')
print("Loading transcriptomic test data...")
test_trn  = pd.read_csv('Data/test/transcriptomic.csv')

print(f"Test patients: {len(test_clin)}")
for name, df in [('Clinical', test_clin), ('Genomic', test_gen),
                 ('Deconv', test_dec), ('ssGSEA', test_ssg),
                 ('Transcriptomic', test_trn)]:
    print(f"  {name:15s} → {df.shape}")

---
# 3. Apply Stored Preprocessing Pipeline

We reproduce the **exact same** steps from the Development Notebook:

| Step | Modality | Transform |
|------|----------|-----------|
| 4.1 | Transcriptomic | Filter same genes → scale (fitted scaler) → PCA (fitted) |
| 4.2 | Mutations | Binary encode MUT/WT/NO_IF |
| 4.3 | Immune deconv. | Drop quality cols → log₁₊ₓ |
| 4.4 | Pathways | Scale (fitted scaler) |
| 4.5 | Clinical | One-hot encode → scale numerics (fitted scaler) |
| 4.6 | All | MICE imputation using fitted kernel |
| 4.7 | Non-trans | Select L1-chosen features |

In [ ]:
# ── 3.1  Transcriptomic ───────────────────────────────────────────────────────
test_trans = test_trn.drop('Patient_ID', axis=1)

# Keep same genes as training
available_genes = [g for g in keep_genes if g in test_trans.columns]
missing_genes   = [g for g in keep_genes if g not in test_trans.columns]
test_trans = test_trans.reindex(columns=keep_genes, fill_value=0)

# Apply fitted scaler
test_trans_scaled = pd.DataFrame(
    trans_scaler.transform(test_trans),
    columns=keep_genes
).fillna(0)

# Apply fitted PCA
test_trans_pca = pd.DataFrame(
    pca.transform(test_trans_scaled),
    columns=[f'PC{i+1}' for i in range(N_COMPONENTS)]
)
print(f"Transcriptomic → {test_trans_pca.shape[1]} PCs  (missing genes filled: {len(missing_genes)})")

In [ ]:
# ── 3.2  Mutations ────────────────────────────────────────────────────────────
test_mut = test_gen.drop('Patient_ID', axis=1)
ensembl_cols = [c for c in test_mut.columns if c.startswith('ENSG')]
test_mut = test_mut.drop(columns=ensembl_cols, errors='ignore')
test_mut = test_mut[mutation_cols]  # align columns
test_mut = test_mut.replace({'MUT': 1, 'WT': 0, 'NO_IF': np.nan}).astype(float)
print(f"Mutations → {test_mut.shape[1]} features")

In [ ]:
# ── 3.3  Immune deconvolution ──────────────────────────────────────────────────
test_deconv = test_dec[cell_type_cols].copy()
test_deconv = np.log1p(test_deconv)
print(f"Immune deconv → {test_deconv.shape[1]} features")

In [ ]:
# ── 3.4  Pathway scores ───────────────────────────────────────────────────────
test_pathway = test_ssg.drop('Patient_ID', axis=1)
test_pathway_scaled = pd.DataFrame(
    pathway_scaler.transform(test_pathway),
    columns=test_pathway.columns
)
print(f"Pathways → {test_pathway_scaled.shape[1]} features")

In [ ]:
# ── 3.5  Clinical ─────────────────────────────────────────────────────────────
test_clinic = test_clin[clinical_feature_cols].copy()
test_clinic = pd.get_dummies(test_clinic, columns=cat_cols, drop_first=True)

# Align columns with training (some categories might be missing in test)
for col in clinical_encoded_cols:
    if col not in test_clinic.columns:
        test_clinic[col] = 0
test_clinic = test_clinic[clinical_encoded_cols]

# Apply fitted scaler to numerical columns
test_clinic[num_cols] = clinical_scaler.transform(test_clinic[num_cols])
print(f"Clinical → {test_clinic.shape[1]} features")

In [ ]:
# ── 3.6  Combine non-transcriptomic & impute ─────────────────────────────────
X_test_small = pd.concat([
    test_clinic.reset_index(drop=True),
    test_mut.reset_index(drop=True),
    test_deconv.reset_index(drop=True),
    test_pathway_scaled.reset_index(drop=True),
], axis=1)

missing_count = X_test_small.isna().sum().sum()
print(f"Missing values in test data: {missing_count}")

if missing_count > 0 and mice_kernel is not None:
    # Use the fitted MICE kernel to impute test data
    test_imputed = mice_kernel.impute_new_data(X_test_small).complete_data()
    X_test_small = test_imputed
    print(f"After MICE imputation: {X_test_small.isna().sum().sum()} missing")
elif missing_count > 0:
    X_test_small = X_test_small.fillna(X_test_small.median())
    print("Filled with median (MICE kernel not available)")

In [ ]:
# ── 3.7  Select L1-chosen features & combine with PCA ────────────────────────
# Ensure all expected columns exist
for col in selected_small_cols:
    if col not in X_test_small.columns:
        X_test_small[col] = 0

X_test_small_sel = X_test_small[selected_small_cols]

X_test_final = pd.concat([
    X_test_small_sel.reset_index(drop=True),
    test_trans_pca.reset_index(drop=True)
], axis=1)

print(f"\n✅ Final test feature matrix: {X_test_final.shape}")

---
# 4. Generate Predictions

In [ ]:
# ── Predict ───────────────────────────────────────────────────────────────────
y_pred = model.predict(X_test_final)
if hasattr(model, 'predict_proba'):
    y_proba = model.predict_proba(X_test_final)[:, 1]
else:
    y_proba = model.decision_function(X_test_final)

predictions_df = pd.DataFrame({
    'Patient_ID': test_clin['Patient_ID'].values,
    'Predicted_Response': y_pred,
    'Probability': np.round(y_proba, 4)
})
display(predictions_df)

print(f"\nPredicted Responders: {y_pred.sum()} / {len(y_pred)}")
print(f"Predicted Non-Resp:   {(1-y_pred).sum()} / {len(y_pred)}")

---
# 5. Performance Report (when labels become available)

Once the instructor releases the true labels, you can evaluate your model
using the cell below.

In [ ]:
# ── Uncomment when true labels are available ──────────────────────────────────
# from sklearn.metrics import (roc_auc_score, average_precision_score,
#                              matthews_corrcoef, confusion_matrix,
#                              ConfusionMatrixDisplay, classification_report)
# import matplotlib.pyplot as plt
#
# y_true = pd.read_csv('Data/test/test_labels.csv')['Response'].values
#
# print("=== Performance Report ===")
# print(f"ROC AUC:  {roc_auc_score(y_true, y_proba):.3f}")
# print(f"PR AUC:   {average_precision_score(y_true, y_proba):.3f}")
# print(f"MCC:      {matthews_corrcoef(y_true, y_pred):.3f}")
# print()
# print(classification_report(y_true, y_pred, target_names=['Non-Resp','Responder']))
#
# fig, ax = plt.subplots(figsize=(5, 4))
# ConfusionMatrixDisplay.from_predictions(y_true, y_pred, ax=ax,
#     display_labels=['Non-Resp', 'Responder'], cmap='Blues')
# ax.set_title('Confusion Matrix – Test Set')
# plt.tight_layout(); plt.show()

---
# 6. Submit Predictions via Google Form

Push your team's predictions directly from this notebook.
No need to open the form manually — the code below sends an HTTP POST request
with your team name and predicted labels.

> ⚠️ **Replace the placeholders** below with the actual Google Form URL
> and entry IDs provided by the instructor.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# ▶  FILL IN THESE VALUES (provided by your instructor)
# ══════════════════════════════════════════════════════════════════════════════
TEAM_NAME      = "YOUR_TEAM_NAME"       # ← change this
FORM_URL       = "https://docs.google.com/forms/d/e/YOUR_FORM_ID/formResponse"
ENTRY_TEAM     = "entry.XXXXXXXXX"       # ← entry ID for Team Name field
ENTRY_PREDS    = "entry.YYYYYYYYY"       # ← entry ID for Predictions field
# ══════════════════════════════════════════════════════════════════════════════

# Convert predictions to a comma-separated string
pred_string = ",".join(map(str, y_pred.tolist()))

payload = {
    ENTRY_TEAM:  TEAM_NAME,
    ENTRY_PREDS: pred_string,
}

response = requests.post(FORM_URL, data=payload)

if response.status_code == 200:
    print(f"✅ Predictions submitted successfully!")
    print(f"   Team: {TEAM_NAME}")
    print(f"   Predictions: {pred_string[:80]}...")
else:
    print(f"❌ Submission failed (HTTP {response.status_code})")
    print(f"   Check the form URL and entry IDs.")

---
## 🎉 Congratulations!

You have successfully completed the full ML pipeline:
1. ✅ Trained a model on multi-modal pharmacogenomic data
2. ✅ Applied the pipeline to unseen test data
3. ✅ Submitted your predictions for evaluation

**Good luck!** 🧬🔬